# Control de contratación directa — contratos para revisión

Este notebook consulta Gold sin modificarlo. Presenta cobertura, bandeja y evidencia de los únicos tres componentes del IRCD. Los resultados permanecen aquí; no exporta gráficos ni tablas. Las reglas son iniciales y no acreditan irregularidades.

**Lectura temporal:** fotografía anual al corte del conjunto descargado. Los datos actualizados de contratos antiguos no reconstruyen lo que se sabía al firmarlos.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'secop_pipeline').exists():
    ROOT = ROOT.parent
if not (ROOT / 'secop_pipeline').exists():
    raise RuntimeError('Abra el notebook desde la carpeta del proyecto o notebooks')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from secop_pipeline.config import GOLD_DIR
path = GOLD_DIR / 'contratos_control' / 'contratos.parquet'
if not path.exists():
    raise RuntimeError('Gold contractual aún no existe. Ejecute 05 o 03 antes de analizar.')
con = duckdb.connect()
con.from_parquet(str(path)).create_view('contratos')
def query(sql, params=None):
    return con.execute(sql, params or []).df()


## 1. Cobertura antes de interpretar alertas

**Por qué:** un índice nulo refleja información insuficiente y debe separarse de un score cero. **Qué y cómo:** contamos contratos por estado, período y clase de persona; mostramos motivos de nulidad. **Conclusión:** las tablas siguientes delimitan qué parte del universo puede evaluarse y por qué el resto no admite una conclusión integral.

In [ ]:
display(query('SELECT rol_periodo,estado_indice,clase_persona,count(*) contratos FROM contratos GROUP BY ALL ORDER BY ALL'))
for comp in ['concentracion','recurrencia','valor_atipico']:
    display(query(f'SELECT motivo_{comp} motivo,count(*) contratos FROM contratos WHERE score_{comp} IS NULL GROUP BY 1 ORDER BY 2 DESC'))
display(query('SELECT fecha_corte,version_parametros,referencia_inicio,referencia_fin,definicion_valor FROM contratos GROUP BY ALL'))


## 2. Distribución de componentes

**Por qué:** antes de priorizar hay que comprobar si las reglas activan casi todos los contratos o ninguno. **Cómo:** barras de frecuencias para scores ordinales 0–2, reservando una categoría para nulos. No se tratan las etiquetas como valores continuos. **Conclusión:** una alerta masiva exige revisar grupos y umbrales; pocos casos también pueden reflejar falta de cobertura.

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4))
for ax, comp in zip(axes,['concentracion','recurrencia','valor_atipico']):
    data=query(f"SELECT coalesce(cast(score_{comp} AS VARCHAR),'sin evidencia') score,count(*) n FROM contratos GROUP BY 1 ORDER BY 1")
    ax.bar(data.score,data.n);ax.set_title(comp);ax.set_ylabel('Contratos');ax.tick_params(axis='x',rotation=25)
plt.tight_layout();plt.show()
display(query('SELECT score_concentracion>0 alerta_c,score_recurrencia>0 alerta_r,score_valor_atipico>0 alerta_a,count(*) contratos FROM contratos GROUP BY ALL'))


## 3. Concentración y recurrencia: posible doble contabilización

**Hipótesis:** una relación repetida entidad–proveedor puede elevar simultáneamente C y R. **Cómo:** Spearman sobre el score C y el máximo score R por entidad–proveedor–año–segmento, usando solo pares completos. Se evita repetir la misma concentración una vez por cada contrato. El scatter muestra participación y recurrencia máxima; no prueba causalidad. **Conclusión:** una asociación elevada indica que el índice puede estar contando parcialmente el mismo patrón; se mantienen los tres pesos exigidos.

In [ ]:
pairs=query('''SELECT entidad_id,proveedor_id,anio,segmento,max(concentracion) c,max(recurrencia) r,
max(score_concentracion) sc,max(score_recurrencia) sr FROM contratos GROUP BY ALL''')
valid=pairs.dropna(subset=['sc','sr'])
print('Pares con scores comparables:',len(valid))
print('Spearman:',valid[['sc','sr']].rank().corr().iloc[0,1] if len(valid)>1 else 'No estimable')
plot=pairs.dropna(subset=['c','r']).sample(n=min(5000,len(pairs.dropna(subset=['c','r']))),random_state=42)
if len(plot):
    plt.scatter(plot.r,plot.c,s=9,alpha=.25)
    plt.xlabel('Máximo de contratos en 365 días del par/año/segmento');plt.ylabel('Participación de recursos (0–1)')
    plt.title('Concentración y recurrencia; muestra reproducible de pares');plt.show()


## 4. Bandeja de revisión

**Por qué:** el producto debe permitir seleccionar contratos. **Cómo:** filtros parametrizados y orden por índice completo y puntuación. Los casos incompletos aparecen identificados; se pueden consultar por componente. No se muestran documentos ni nombres personales en la vista general. Las señales sirven para verificar expedientes.

In [ ]:
anio = 2025
entidad = None   # entidad_id exacto, o None
persona = None   # 'natural', 'indeterminado', o None
componente = None  # 'concentracion', 'recurrencia', 'valor_atipico', o None
where=['anio=?'];params=[anio]
if entidad is not None: where.append('entidad_id=?');params.append(entidad)
if persona is not None: where.append('clase_persona=?');params.append(persona)
if componente is not None:
    if componente not in ('concentracion','recurrencia','valor_atipico'): raise ValueError('Componente inválido')
    where.append('score_'+componente+'>0')
bandeja=query('''SELECT id_contrato,nombre_entidad,fecha_de_firma,causal,tipo_de_contrato,valor_del_contrato,
score_concentracion,score_recurrencia,score_valor_atipico,ircd,estado_indice,nivel_revision,raw__urlproceso
FROM contratos WHERE '''+' AND '.join(where)+' ORDER BY ircd DESC NULLS LAST,id_contrato LIMIT 100',params)
display(bandeja)


## 5. Ficha explicable de un contrato

**Qué:** entradas, cálculos, grupos, umbrales, motivos de nulo y contratos de la misma ventana. **Cómo:** selección por ID exacto; el listado de recurrencia respeta entidad, proveedor, segmento y ambas fechas inclusivas. **Conclusión:** contraste las señales con el expediente y documente la explicación antes de decidir una revisión. Un score no sustituye ese examen.

In [ ]:
contrato_id = None  # Copiar un ID de la bandeja
if contrato_id:
    ficha=query('SELECT * FROM contratos WHERE id_contrato=?',[contrato_id])
    if ficha.empty: print('Contrato no encontrado')
    else:
        cols=['id_contrato','fecha_de_firma','causal','tipo_de_contrato','codigo_de_categoria_principal','departamento',
        'valor_del_contrato','duracion_dias','numerador_proveedor','denominador_entidad','concentracion','n_entidad_segmento',
        'estrato_volumen','n_ref_c','p90_c','p95_c','ventana_inicio','ventana_fin','recurrencia','n_ref_r','p90_r','p95_r',
        'valor_mensual','mediana_a','ratio_valor','n_ref_a','p90_a','p95_a','anio_referencia_valor',
        'motivo_concentracion','motivo_recurrencia','motivo_valor_atipico','score_concentracion','score_recurrencia','score_valor_atipico',
        'ircd','estado_indice','nivel_revision','fecha_corte','fecha_calculo','version_parametros','huella_parametros',
        'filtro_personas','referencia_inicio','referencia_fin','modo_temporal','definicion_valor','fuente_archivo','huella_registro','raw__urlproceso']
        display(ficha[cols].T)
        display(query('''SELECT b.id_contrato,b.fecha_de_firma,b.valor_del_contrato FROM contratos a JOIN contratos b
        ON a.entidad_id=b.entidad_id AND a.proveedor_id=b.proveedor_id AND a.segmento=b.segmento
        AND b.fecha_de_firma::DATE BETWEEN a.ventana_inicio AND a.ventana_fin
        WHERE a.id_contrato=? ORDER BY b.fecha_de_firma,b.id_contrato''',[contrato_id]))
else:
    print('Seleccione un contrato; no se presenta evidencia individual por defecto.')


## 6. Revisión de calibración

Compare 2025 (validación) con 2026 (seguimiento) sin recalibrar con datos futuros. Revise empates P90=P95 de recurrencia y la proporción que se activaría. Revise expedientes con y sin alertas; registre hipótesis alternativas de renovación legítima, especialidad del servicio y estructura de contratación. Las conclusiones sustantivas quedan pendientes de ejecutar sobre el nuevo Gold y revisar documentos.

In [ ]:
display(query('''SELECT rol_periodo,count(*) contratos,count(ircd) indices_completos,
count(*) FILTER(WHERE p90_r=p95_r) empates_r,
count(*) FILTER(WHERE p90_r=p95_r AND recurrencia>=2 AND recurrencia>=p90_r) alertables_si_aceptar_empates
FROM contratos GROUP BY 1'''))


## Continuación: réplica descriptiva de Fedesarrollo

Abra [08_eda_fedesarrollo_control.ipynb](08_eda_fedesarrollo_control.ipynb). Parte de Silver, permite explorar el caso aunque aún no exista Gold contractual y documenta qué análisis del artículo pueden replicarse con las fuentes locales. Los resultados quedan en sus celdas; el IRCD no cambia.